# GODMAX xDESI Stage-31 Local HMC Debug Run

This notebook runs the fixed-cosmology Stage-31 sampler interactively, without Slurm. It is intentionally verbose: each expensive step prints a timestamp, reports the JAX backend, checks whether the fiducial model value is finite, checks whether NumPyro can initialize gradients, and only then runs NUTS.

If initialization fails, inspect the diagnostic cells before running the sampler cell. The current known failure mode is finite likelihood value but invalid gradients through the GODMAX/HOD theory path.


In [22]:
# This cell must run before importing JAX in a fresh kernel.
# Restart the kernel first if JAX was already imported and you want these settings to apply.
import os

USE_GPU = True
GPU_MEM_FRACTION = "0.95"
RUN_GPU_SANITY_CHECK = True
GPU_SANITY_MATRIX_SIZE = 4096

if USE_GPU:
    os.environ["JAX_PLATFORMS"] = "cuda"
else:
    os.environ["JAX_PLATFORMS"] = "cpu"

print("Using GPU:", USE_GPU)

os.environ["JAX_ENABLE_X64"] = "True"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = GPU_MEM_FRACTION

print("JAX_PLATFORMS=", os.environ.get("JAX_PLATFORMS"))
print("XLA_PYTHON_CLIENT_PREALLOCATE=", os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"))
print("XLA_PYTHON_CLIENT_MEM_FRACTION=", os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION"))


Using GPU: True
JAX_PLATFORMS= cuda
XLA_PYTHON_CLIENT_PREALLOCATE= true
XLA_PYTHON_CLIENT_MEM_FRACTION= 0.95


In [23]:
from __future__ import annotations

import json
import os
import sys
import time
import traceback
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "param_files").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(f"Could not find GODMAX repo root from {start}")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
SURVEY_DIR = REPO_ROOT / "notebooks" / "xDESI" / "survey_measure"
if str(SURVEY_DIR) not in sys.path:
    sys.path.insert(0, str(SURVEY_DIR))

print("repo root:", REPO_ROOT)
print("survey dir:", SURVEY_DIR)

# try:
get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")
# except Exception:
#     pass


repo root: /mnt/ceph/users/spandey/ltu-godmax/GODMAX
survey dir: /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/survey_measure
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:

import jax
import jax.numpy as jnp
import numpyro

import godmax_multiprobe_hmc_stage31 as hmc31

PLATFORM = "gpu" if USE_GPU else "cpu"
runtime = hmc31.configure_numpyro_platform(PLATFORM)
print(json.dumps(runtime, indent=2))
print("jax_enable_x64:", jax.config.jax_enable_x64)

if RUN_GPU_SANITY_CHECK:
    gpu_check = hmc31.gpu_sanity_check(GPU_SANITY_MATRIX_SIZE, require_gpu=USE_GPU)
    print("gpu sanity check")
    print(json.dumps(hmc31.gmt.to_jsonable(gpu_check), indent=2))


{
  "requested_platform": "gpu",
  "jax_default_backend": "gpu",
  "jax_device_count": 1,
  "jax_device_platforms": [
    "gpu"
  ],
  "jax_devices": [
    "cuda:0"
  ]
}
jax_enable_x64: True
gpu sanity check
{
  "jax_default_backend": "gpu",
  "jax_device_platforms": [
    "gpu"
  ],
  "jax_devices": [
    "cuda:0"
  ],
  "matrix_size": 4096,
  "result": 68719476736.0,
  "seconds": 0.00593113899230957
}


In [25]:

def now() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def mark(message: str) -> None:
    print(f"[{now()}] {message}", flush=True)


def run_step(label: str, func):
    mark(f"START {label}")
    t0 = time.time()
    try:
        out = func()
    except Exception as exc:
        mark(f"FAILED {label} after {time.time() - t0:.1f} s: {type(exc).__name__}: {exc}")
        traceback.print_exc(limit=12)
        raise
    mark(f"DONE {label} in {time.time() - t0:.1f} s")
    return out


## Controls

Use `RUN_MODE = "smoke"` for an interactive test. Switch to `"full"` only after the initialization diagnostics are clean. You can override any sampler setting below without editing the YAML.


In [26]:
CONFIG = REPO_ROOT / "param_files" / "xDESI" / "params_multiprobe_fast1024_hmc_stage31.yaml"
RUN_MODE = "smoke"  # "smoke" or "full"
SMOKE = RUN_MODE == "smoke"

# Keep max_tree_depth fixed at 4 unless deliberately testing something else.
SAMPLER_OVERRIDES = {
    # "seed": 44,
    # "num_warmup": 5,
    # "num_samples": 5,
    # "num_chains": 1,
    # "target_accept_prob": 0.8,
    # "progress_bar": True,
}

REQUIRE_VALID_INIT_BEFORE_NUTS = True
RUN_FIDUCIAL_WINDOWING_CHECK = False
RUN_FIDUCIAL_VALUE_CHECK = False
RUN_INITIALIZATION_DIAGNOSTIC = False
RUN_PHYSICAL_GRAD_DIAGNOSTIC = False
SAVE_OUTPUTS_AFTER_SAMPLING = True
OUTPUT_DIR = REPO_ROOT / "notebooks" / "xDESI" / "survey_measure" / "outputs" / "godmax_multiprobe_fast1024_true_nz_hmc_stage31_local"

print("config:", CONFIG)
print("run mode:", RUN_MODE)
print("output dir:", OUTPUT_DIR)

config: /mnt/ceph/users/spandey/ltu-godmax/GODMAX/param_files/xDESI/params_multiprobe_fast1024_hmc_stage31.yaml
run mode: smoke
output dir: /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/survey_measure/outputs/godmax_multiprobe_fast1024_true_nz_hmc_stage31_local


## Build Context And Static Checks


In [27]:

context = run_step("prepare_fit_context", lambda: hmc31.prepare_fit_context(CONFIG))
summary = hmc31.static_summary(context)
print(json.dumps(hmc31.gmt.to_jsonable(summary), indent=2))

settings = hmc31.sampler_settings(context.stage_config, smoke=SMOKE, overrides=SAMPLER_OVERRIDES)
print("sampler settings")
print(json.dumps(hmc31.gmt.to_jsonable(settings), indent=2))


[2026-06-04 00:15:21] START prepare_fit_context


[2026-06-04 00:15:22] DONE prepare_fit_context in 1.1 s
{
  "stage": "stage31_legacy_core",
  "n_parameters": 31,
  "n_spectra": 46,
  "data_vector_size": 460,
  "covariance_rank": 459,
  "covariance_size": 460,
  "dropped_covariance_modes": 1,
  "corr_eigenvalue_threshold": 1e-08,
  "min_corr_eigenvalue": -0.0036058517876595567,
  "max_corr_eigenvalue": 4.9400153738071,
  "parameter_names": [
    "log10_Mstar0_theta_ej",
    "theta_ej_0",
    "nu_theta_ej_M",
    "nu_theta_ej_z",
    "log10_Mc0",
    "mu_beta",
    "alpha_nt",
    "log10M1_fshmr_pz1",
    "log10M1_fshmr_pz2",
    "log10M1_fshmr_pz3",
    "log10M1_fshmr_pz4",
    "log10M1_a_fshmr_pz1",
    "log10M1_a_fshmr_pz2",
    "log10M1_a_fshmr_pz3",
    "log10M1_a_fshmr_pz4",
    "delta_fshmr_pz1",
    "delta_fshmr_pz2",
    "delta_fshmr_pz3",
    "delta_fshmr_pz4",
    "gamma_fshmr_pz1",
    "gamma_fshmr_pz2",
    "gamma_fshmr_pz3",
    "gamma_fshmr_pz4",
    "siglogMstar_Ncen_pz1",
    "siglogMstar_Ncen_pz2",
    "siglogMstar_N

In [28]:
spec_rows = [
    {
        "i": i,
        "name": spec.name,
        "target": spec.target,
        "fiducial": spec.fiducial,
        "prior_min": spec.prior_min,
        "prior_max": spec.prior_max,
        "inside_prior": spec.prior_min <= spec.fiducial <= spec.prior_max,
    }
    for i, spec in enumerate(context.parameter_specs)
]
spec_table = pd.DataFrame(spec_rows)
spec_table


,i,name,target,fiducial,prior_min,prior_max,inside_prior
0,0,log10_Mstar0_theta_ej,sim_scalar,15.0000,14.000,16.5,True
1,1,theta_ej_0,sim_scalar,2.0000,0.500,8.0,True
2,2,nu_theta_ej_M,sim_scalar,-0.3300,-1.000,1.0,True
3,3,nu_theta_ej_z,sim_scalar,-1.2200,-2.000,2.0,True
4,4,log10_Mc0,sim_scalar,13.7500,13.000,16.0,True
5,5,mu_beta,sim_scalar,0.0762,0.005,1.5,True
6,6,alpha_nt,sim_scalar,0.0500,0.000,0.5,True
7,7,log10M1_fshmr_pz1,hod_array,14.5913,11.000,15.5,True
8,8,log10M1_fshmr_pz2,hod_array,14.6232,11.000,15.5,True
9,9,log10M1_fshmr_pz3,hod_array,14.5384,11.000,15.5,True


## DESI true n(z), photo-z diagnostic, and HOD number-density targets

This cell shows the calibrated true-redshift DESI `dN/dz` used in theory, the catalog `Z_PHOT_MEDIAN` histogram saved only as a diagnostic, the per-pz comoving number density target, and the corresponding physical/proper number density. Each pz bin is fit with its own single-pz GODMAX galaxy object; no HOD parameter is assigned by true-redshift support intervals.

In [29]:
import matplotlib.pyplot as plt

md_nbar = context.config["metadata"]
z_nbar = np.asarray(md_nbar["lens_z_mid"])
dndz_pz = np.asarray(md_nbar["lens_dndz"])
photoz = md_nbar.get("lens_photoz_diagnostic") or {}
z_photo = None if photoz.get("z_mid") is None else np.asarray(photoz["z_mid"])
dndz_photo = None if photoz.get("dndz_by_pz") is None else np.asarray(photoz["dndz_by_pz"])
nbar_comoving_by_pz = np.asarray(md_nbar["nbar_comoving_by_pz"])
nbar_physical_by_pz = np.asarray(md_nbar["nbar_physical_by_pz"])
nbar_sum = np.asarray(md_nbar["nbar_comoving_sum_by_pz"])
nbar_per_sr = np.asarray(md_nbar["nbar_per_sr_by_pz"])
chi = np.asarray(md_nbar["chi_lens_hmpc"])
dchi_dz = np.asarray(md_nbar["dchi_dz_lens_hmpc"])

rows = []
for i in range(4):
    mask = dndz_pz[i] > 0
    recovered_nbar_sr = np.trapz(nbar_comoving_by_pz[i] * chi**2 * dchi_dz, z_nbar)
    rows.append(
        {
            "pz": i + 1,
            "nbar_per_sr_measured": nbar_per_sr[i],
            "nbar_per_sr_recovered": recovered_nbar_sr,
            "fractional_recovery_error": (recovered_nbar_sr / nbar_per_sr[i] - 1.0),
            "true_dndz_integral": np.trapz(dndz_pz[i], z_nbar),
            "true_mean_z": np.trapz(z_nbar * dndz_pz[i], z_nbar),
            "photoz_mean_z": np.trapz(z_photo * dndz_photo[i], z_photo) if z_photo is not None else np.nan,
            "z_support_min": z_nbar[mask][0] if np.any(mask) else np.nan,
            "z_support_max": z_nbar[mask][-1] if np.any(mask) else np.nan,
            "comoving_median": np.median(nbar_comoving_by_pz[i, mask]) if np.any(mask) else np.nan,
            "physical_median": np.median(nbar_physical_by_pz[i, mask]) if np.any(mask) else np.nan,
        }
    )
nbar_summary_table = pd.DataFrame(rows)
display(nbar_summary_table)
print("nbar target convention:", md_nbar.get("nbar_target_convention"))
print("DESI lens redshift kind:", md_nbar.get("lens_redshift_kind"))
print("DESI lens n(z) provenance:", json.dumps(hmc31.gmt.to_jsonable(md_nbar.get("lens_nz_provenance", {})), indent=2))

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
axes = axes.ravel()

for i in range(4):
    color = colors[i % len(colors)]
    axes[0].plot(z_nbar, dndz_pz[i], color=color, lw=2.0, label=f"pz{i + 1} true")
    if z_photo is not None:
        axes[1].plot(z_photo, dndz_photo[i], color=color, lw=1.6, label=f"pz{i + 1} photo-z")
    axes[2].plot(z_nbar, nbar_comoving_by_pz[i], color=color, label=f"pz{i + 1}")
    axes[3].plot(z_nbar, nbar_physical_by_pz[i], color=color, label=f"pz{i + 1}")

axes[2].plot(z_nbar, nbar_sum, color="0.45", ls=":", lw=2.0, label="sum over pz")
axes[0].set_ylabel("true normalized dN/dz")
axes[1].set_ylabel("photo-z diagnostic dN/dz")
axes[2].set_ylabel("comoving nbar [(Mpc/h)^-3]")
axes[3].set_ylabel("physical nbar [(Mpc/h)^-3 proper]")
for ax in axes:
    ax.set_xlabel("z")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=9)
plt.show()

,pz,nbar_per_sr_measured,nbar_per_sr_recovered,fractional_recovery_error,true_dndz_integral,true_mean_z,photoz_mean_z,z_support_min,z_support_max,comoving_median,physical_median
0,1,4.964671e+05,4.964677e+05,1.318276e-06,1.0,0.468044,0.471306,0.07,1.05,0.000018,0.000033
1,2,8.426690e+05,8.426695e+05,5.111347e-07,1.0,0.632793,0.634992,0.07,1.39,0.000013,0.000033
2,3,1.099294e+06,1.099294e+06,2.689364e-07,1.0,0.791285,0.796454,0.05,1.41,0.000009,0.000046
3,4,1.006018e+06,1.006018e+06,1.114246e-07,1.0,0.925941,0.918186,0.09,1.43,0.000006,0.000040


nbar target convention: per_photoz_bin_single_godmax_models
DESI lens redshift kind: spectroscopic_calibrated_true_redshift
DESI lens n(z) provenance: {
  "ksz_sigma_true_gas_calibration_json": "{\n  \"bin_assignment\": \"by_z_photo\",\n  \"c_km_s_convention\": 300000.0,\n  \"corrcoef_v_rec_v_true\": {\n    \"1\": 0.6871109287591863,\n    \"2\": 0.6683324285379622,\n    \"3\": 0.6210174474653414,\n    \"4\": 0.4660139572002138\n  },\n  \"description\": \"sigma_true_gas calibration for the DESI DR10 extended photometric tomographic bins in the ACTxDESI transfer package.\",\n  \"doc\": \"data/xDESI/survey_data/docs/DESI_ABACUS_SIGMA_TRUE_GAS.md\",\n  \"n_objects\": {\n    \"1\": 3279180,\n    \"2\": 3362292,\n    \"3\": 1304255,\n    \"4\": 360183\n  },\n  \"paper_definition\": \"Eq. 21-22: sigma_true denotes the velocity dispersion of the true line-of-sight velocities obtained from AbacusSummit. Eq. 37 notes the real-space stacking convention measures tau_bar * sigma_true / c.\",\n  \"p

## Fiducial Theory And Windowing Check

This builds the GODMAX fiducial model once and checks the JAX-native data vector against the existing measurement wrapper. If this fails, do not run sampling.


In [30]:

if RUN_FIDUCIAL_WINDOWING_CHECK:
    fid_check = run_step("compare_fiducial_windowing", lambda: hmc31.compare_fiducial_windowing(context))
    print("allclose:", fid_check["allclose"])
    print("max_abs_delta:", fid_check["max_abs_delta"])
    print("max_rel_delta:", fid_check["max_rel_delta"])
    print("raw kSZ median C_ell by pz:", fid_check["ksz_median_raw_by_pz"])
else:
    fid_check = None
    print("RUN_FIDUCIAL_WINDOWING_CHECK=False; skipped wrapper/windowing comparison.")


RUN_FIDUCIAL_WINDOWING_CHECK=False; skipped wrapper/windowing comparison.


In [31]:

def fiducial_value_check():
    fid = hmc31.pack_fiducial_sample(context.parameter_specs)
    theory = hmc31.evaluate_sample_theory_vector(context, fid)
    chi2 = hmc31.whitened_chi2(context.likelihood, theory)
    theory_np = np.asarray(theory)
    return {
        "theory_size": int(theory_np.size),
        "theory_finite": bool(np.all(np.isfinite(theory_np))),
        "theory_nan_count": int(np.sum(~np.isfinite(theory_np))),
        "chi2": float(np.asarray(chi2)),
        "chi2_finite": bool(np.isfinite(np.asarray(chi2))),
        "theory_min": float(np.nanmin(theory_np)),
        "theory_max": float(np.nanmax(theory_np)),
    }

if RUN_FIDUCIAL_VALUE_CHECK:
    fid_value_report = run_step("fiducial finite-value check", fiducial_value_check)
    print(json.dumps(fid_value_report, indent=2))
else:
    fid_value_report = None
    print("RUN_FIDUCIAL_VALUE_CHECK=False; skipped fiducial finite-value check.")


RUN_FIDUCIAL_VALUE_CHECK=False; skipped fiducial finite-value check.


## NumPyro Initialization Diagnostic

This is the same check that NUTS performs before sampling. `validate_grad=False` only checks the model value. `validate_grad=True` also checks gradients. NUTS needs the second check to pass.


In [32]:

if RUN_INITIALIZATION_DIAGNOSTIC:
    init_report = run_step("numpyro initialize_model diagnostics", lambda: hmc31.initialization_diagnostics(context, settings))
    print(json.dumps(hmc31.gmt.to_jsonable(init_report), indent=2))
    valid_init = all(check.get("ok", False) for check in init_report["checks"])
else:
    init_report = {"checks": []}
    valid_init = True
    print("RUN_INITIALIZATION_DIAGNOSTIC=False; skipping pre-NUTS initialization diagnostic.")

print("valid_init_for_nuts:", valid_init)


RUN_INITIALIZATION_DIAGNOSTIC=False; skipping pre-NUTS initialization diagnostic.
valid_init_for_nuts: True


## Direct Physical-Parameter Gradient Diagnostic

This bypasses NumPyro's constrained-site initialization wrapper and differentiates the whitened chi-square directly with respect to the 31 physical parameters. If this reports finite gradients but NumPyro init fails, the problem is likely in NumPyro's Uniform transform. If this reports bad gradients, the issue is in the theory/likelihood path itself.

In [33]:
if RUN_PHYSICAL_GRAD_DIAGNOSTIC:
    physical_grad_report = run_step(
        "physical parameter value_and_grad diagnostics",
        lambda: hmc31.physical_gradient_diagnostics(context),
    )
    print(json.dumps(hmc31.gmt.to_jsonable(physical_grad_report), indent=2))
    if physical_grad_report.get("bad_gradients"):
        display(pd.DataFrame(physical_grad_report["bad_gradients"]))
else:
    physical_grad_report = None
    print("RUN_PHYSICAL_GRAD_DIAGNOSTIC=False; skipped direct physical gradient diagnostic.")

RUN_PHYSICAL_GRAD_DIAGNOSTIC=False; skipped direct physical gradient diagnostic.


## Optional Finite-Difference Value Scan

Run this when gradients fail. It checks whether small positive/negative moves in selected physical parameters still give finite chi-square values. Finite differences being well behaved while autodiff fails points to a differentiability or JAX tracing issue rather than an invalid model value.

In [34]:
RUN_FINITE_DIFFERENCE_SCAN = False
FINITE_DIFFERENCE_PARAMETER_INDICES = list(range(len(context.parameter_specs)))

if RUN_FINITE_DIFFERENCE_SCAN:
    fd_rows = run_step(
        "finite-difference scan",
        lambda: hmc31.finite_difference_diagnostics(
            context,
            parameter_indices=FINITE_DIFFERENCE_PARAMETER_INDICES,
            rel_step=1.0e-4,
            abs_step=1.0e-5,
        ),
    )
    fd_table = pd.DataFrame(fd_rows)
    display(fd_table)
else:
    print("Set RUN_FINITE_DIFFERENCE_SCAN=True and rerun this cell to scan finite differences.")

Set RUN_FINITE_DIFFERENCE_SCAN=True and rerun this cell to scan finite differences.


## Optional Per-Parameter JVP Gradient Scan

Run this when initialization fails. It probes the derivative of the whitened chi-square with respect to each physical parameter at the fiducial point using one forward-mode directional derivative at a time. This is expensive because each row can trace/evaluate the theory path.


In [35]:
RUN_JVP_SCAN = False
JVP_PARAMETER_INDICES = list(range(len(context.parameter_specs)))

if RUN_JVP_SCAN:
    jvp_rows = run_step(
        "per-parameter JVP scan",
        lambda: hmc31.physical_jvp_diagnostics(context, parameter_indices=JVP_PARAMETER_INDICES),
    )
    jvp_table = pd.DataFrame(jvp_rows)
    display(jvp_table)
else:
    print("Set RUN_JVP_SCAN=True and rerun this cell to scan forward-mode gradients parameter by parameter.")

Set RUN_JVP_SCAN=True and rerun this cell to scan forward-mode gradients parameter by parameter.


## Run NUTS Locally

This cell runs the same sampler machinery as the CLI but stays inside the notebook. It catches and prints the full error. If `REQUIRE_VALID_INIT_BEFORE_NUTS=True`, the cell stops before NUTS when the init diagnostic has already failed.


In [36]:
if REQUIRE_VALID_INIT_BEFORE_NUTS and not valid_init:
    raise RuntimeError(
        "Skipping NUTS because initialization gradient validation failed. "
        "Inspect init_report and optionally run the JVP scan above. "
        "Set REQUIRE_VALID_INIT_BEFORE_NUTS=False if you explicitly want to reproduce the NUTS failure."
    )

mcmc = run_step("run_hmc", lambda: hmc31.run_hmc(context, smoke=SMOKE, overrides=SAMPLER_OVERRIDES))
mcmc.print_summary()

extra = {k: np.asarray(v) for k, v in mcmc.get_extra_fields().items()}
print("extra fields:", {k: v.shape for k, v in extra.items()})
if "diverging" in extra:
    print("divergences:", int(np.sum(extra["diverging"])))
if "accept_prob" in extra:
    print("mean accept_prob:", float(np.mean(extra["accept_prob"])))
if "num_steps" in extra:
    print("num_steps min/median/max:", np.min(extra["num_steps"]), np.median(extra["num_steps"]), np.max(extra["num_steps"]))


[2026-06-04 00:15:25] START run_hmc
[2026-06-04 00:15:25] [hmc] configuring NUTS num_chains=1 chain_method=sequential num_warmup=80 num_samples=800 max_tree_depth=4 target_accept_prob=0.8 dense_mass=True progress_bar=True
[2026-06-04 00:15:25] [hmc] mcmc.run begin


sample: 100%|██████████| 880/880 [15:04<00:00,  1.03s/it, 7 steps of size 2.82e-02. acc. prob=0.25]  


[2026-06-04 00:46:32] [hmc] mcmc.run done
[2026-06-04 00:46:32] DONE run_hmc in 1866.4 s

                             mean       std    median      5.0%     95.0%     n_eff     r_hat
               alpha_nt      0.06      0.00      0.06      0.06      0.06     15.33      1.01
      alphasat_Nsat_pz1      0.56      0.00      0.56      0.56      0.57     11.23      1.00
      alphasat_Nsat_pz2      0.91      0.01      0.91      0.89      0.93     26.02      1.01
      alphasat_Nsat_pz3      1.16      0.03      1.17      1.08      1.18      4.41      1.45
      alphasat_Nsat_pz4      1.49      0.04      1.51      1.43      1.51      6.17      1.22
        delta_fshmr_pz1      0.66      0.01      0.66      0.64      0.66      6.92      1.14
        delta_fshmr_pz2      0.85      0.01      0.85      0.84      0.86      6.51      1.27
        delta_fshmr_pz3      0.66      0.01      0.66      0.64      0.67      6.97      1.28
        delta_fshmr_pz4      0.64      0.01      0.65      0.62 

## Save Outputs


In [37]:

if "mcmc" not in globals():
    raise RuntimeError("No `mcmc` object exists yet. Run the sampler cell first.")

if SAVE_OUTPUTS_AFTER_SAMPLING:
    saved = run_step("save_fit_outputs", lambda: hmc31.save_fit_outputs(context, mcmc, smoke=SMOKE, output_dir=OUTPUT_DIR))
    print(json.dumps({k: str(v) if k != "plots" else [str(p) for p in v] for k, v in saved.items()}, indent=2))
else:
    print("SAVE_OUTPUTS_AFTER_SAMPLING=False; outputs not written.")


[2026-06-04 00:46:32] START save_fit_outputs
[2026-06-04 00:46:32] [hmc] saving outputs begin


[2026-06-04 00:49:20] [hmc] saving outputs done best_chi2=9.10108050e+03
[2026-06-04 00:49:20] DONE save_fit_outputs in 167.8 s
{
  "chain": "/mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/survey_measure/outputs/godmax_multiprobe_fast1024_true_nz_hmc_stage31_local/chain_smoke_stage31.npz",
  "bestfit_params": "/mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/survey_measure/outputs/godmax_multiprobe_fast1024_true_nz_hmc_stage31_local/bestfit_params_smoke_stage31.yaml",
  "bestfit_theory_vector": "/mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/survey_measure/outputs/godmax_multiprobe_fast1024_true_nz_hmc_stage31_local/bestfit_theory_data_vector_smoke_stage31.npz",
  "summary": "/mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/survey_measure/outputs/godmax_multiprobe_fast1024_true_nz_hmc_stage31_local/fit_summary_smoke_stage31.json",
  "pdf": "/mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/survey_measure/outputs/godmax_multiprobe_fast1024

## Posterior Quick Look


In [38]:

if "mcmc" not in globals():
    raise RuntimeError("No `mcmc` object exists yet. Run the sampler cell first.")

samples = mcmc.get_samples(group_by_chain=False)
rows = []
for spec in context.parameter_specs:
    arr = np.asarray(samples[spec.name])
    rows.append(
        {
            "name": spec.name,
            "mean": float(np.mean(arr)),
            "std": float(np.std(arr)),
            "p16": float(np.percentile(arr, 16)),
            "p50": float(np.percentile(arr, 50)),
            "p84": float(np.percentile(arr, 84)),
            "fiducial": spec.fiducial,
            "prior_min": spec.prior_min,
            "prior_max": spec.prior_max,
        }
    )
posterior_table = pd.DataFrame(rows)
posterior_table


,name,mean,std,p16,p50,p84,fiducial,prior_min,prior_max
0,log10_Mstar0_theta_ej,15.368857,0.018571,15.365991,15.372991,15.382737,15.0000,14.000,16.5
1,theta_ej_0,2.984818,0.106476,2.933461,3.031674,3.042026,2.0000,0.500,8.0
2,nu_theta_ej_M,-0.733409,0.024415,-0.745457,-0.743686,-0.718491,-0.3300,-1.000,1.0
3,nu_theta_ej_z,-0.919408,0.018794,-0.932442,-0.912972,-0.908188,-1.2200,-2.000,2.0
4,log10_Mc0,13.737588,0.005678,13.731902,13.737067,13.741899,13.7500,13.000,16.0
5,mu_beta,0.072525,0.000788,0.071858,0.072279,0.073473,0.0762,0.005,1.5
6,alpha_nt,0.058659,0.000560,0.058354,0.058719,0.058991,0.0500,0.000,0.5
7,log10M1_fshmr_pz1,13.860460,0.015866,13.853628,13.861811,13.874080,14.5913,11.000,15.5
8,log10M1_fshmr_pz2,14.418669,0.020560,14.408745,14.425544,14.431929,14.6232,11.000,15.5
9,log10M1_fshmr_pz3,13.671433,0.022151,13.649492,13.665282,13.696971,14.5384,11.000,15.5


## Command-Line Equivalents

These are useful only after the notebook diagnostics show that initialization is valid.


In [39]:

PYTHON = "/mnt/home/spandey/miniconda3/envs/ili-sbi/bin/python"
RUNNER = REPO_ROOT / "notebooks" / "xDESI" / "survey_measure" / "run_godmax_multiprobe_hmc_stage31.py"
print(f"{PYTHON} {RUNNER} --platform gpu --smoke --debug-init")
print(f"{PYTHON} {RUNNER} --platform gpu --debug-init")


/mnt/home/spandey/miniconda3/envs/ili-sbi/bin/python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/survey_measure/run_godmax_multiprobe_hmc_stage31.py --platform gpu --smoke --debug-init
/mnt/home/spandey/miniconda3/envs/ili-sbi/bin/python /mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/xDESI/survey_measure/run_godmax_multiprobe_hmc_stage31.py --platform gpu --debug-init
